<!-- NB_DOC_INTRO_V1 -->
# NLP — Information Retrieval (BM25, embeddings, hybrid)

> 📚 **Doc thematique** : [docs/05_NLP.md](docs/05_NLP.md) (NLP)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Information Retrieval** = retrouver les documents les plus pertinents pour une requete. Ce notebook execute les trois familles : **lexicale** (BM25), **semantique** (embeddings), **hybride** (RRF). Pour le RAG complet : [NLP_LangChain_RAG.ipynb](./NLP_LangChain_RAG.ipynb).

## Plan

1. Corpus + queries jouet
2. BM25 (lexical, baseline robuste)
3. Embeddings semantiques (sentence-transformers)
4. Hybrid search (BM25 + dense + Reciprocal Rank Fusion)
5. Eval (NDCG@k, MRR)
6. Pieges + References


In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Corpus jouet

In [ ]:
docs = [
    "Python is a high-level programming language for general-purpose use.",
    "JavaScript is the language of the web browsers.",
    "Pandas is a Python library for data analysis and manipulation.",
    "NumPy provides efficient array operations for numerical computing.",
    "Scikit-learn is a popular machine learning library in Python.",
    "TensorFlow and PyTorch are deep learning frameworks.",
    "FastAPI is a modern Python web framework for building APIs.",
    "React is a JavaScript library for building user interfaces.",
    "SQL is a query language for relational databases.",
    "Docker simplifies application packaging and deployment.",
]
queries = [
    "Python data analysis library",
    "Machine learning framework",
    "Web programming",
]
print(f"Corpus : {len(docs)} docs, queries : {len(queries)}")

## 2. BM25 — baseline lexicale

In [ ]:
from rank_bm25 import BM25Okapi

# Tokenization simple (lowercase + split)
tokenized = [d.lower().split() for d in docs]
bm25 = BM25Okapi(tokenized)

def search_bm25(query, k=3):
    scores = bm25.get_scores(query.lower().split())
    top_ids = np.argsort(scores)[::-1][:k]
    return [(docs[i], scores[i]) for i in top_ids if scores[i] > 0]

for q in queries:
    print(f"\nBM25 : {q}")
    for doc, sc in search_bm25(q):
        print(f"  [{sc:.3f}] {doc[:80]}")

## 3. Embeddings semantiques — sentence-transformers

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity

    print("Loading MiniLM (~ 80 MB)...")
    embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    doc_embs = embedder.encode(docs, show_progress_bar=False)

    def search_dense(query, k=3):
        q_emb = embedder.encode([query])
        sims = cosine_similarity(q_emb, doc_embs)[0]
        top_ids = np.argsort(sims)[::-1][:k]
        return [(docs[i], sims[i]) for i in top_ids]

    for q in queries:
        print(f"\nDense : {q}")
        for doc, sc in search_dense(q):
            print(f"  [{sc:.3f}] {doc[:80]}")
    DENSE_OK = True
except ImportError:
    print("sentence-transformers pas installe")
    DENSE_OK = False

## 4. Hybrid search — RRF (Reciprocal Rank Fusion)

In [ ]:
if DENSE_OK:
    def hybrid_search(query, k=3, k_rrf=60):
        # BM25 ranks
        bm25_scores = bm25.get_scores(query.lower().split())
        bm25_ranks = sorted(range(len(docs)), key=lambda i: -bm25_scores[i])

        # Dense ranks
        q_emb = embedder.encode([query])
        sims = cosine_similarity(q_emb, doc_embs)[0]
        dense_ranks = sorted(range(len(docs)), key=lambda i: -sims[i])

        # RRF : score = sum(1 / (k + rank))
        rrf_scores = {}
        for rk_list in [bm25_ranks, dense_ranks]:
            for rank, doc_id in enumerate(rk_list, 1):
                rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (k_rrf + rank)
        top = sorted(rrf_scores.items(), key=lambda x: -x[1])[:k]
        return [(docs[i], s) for i, s in top]

    for q in queries:
        print(f"\nHybrid : {q}")
        for doc, sc in hybrid_search(q):
            print(f"  [RRF={sc:.4f}] {doc[:80]}")
else:
    print("SKIP hybrid (dense indispo)")

## 5. Eval — quand on a un dataset gold

```python
from sklearn.metrics import ndcg_score
# y_true : shape (n_queries, n_docs), gold relevance scores 0-3
# y_score : shape (n_queries, n_docs), predicted scores
ndcg_at_5 = ndcg_score(y_true, y_score, k=5)
```

Voir [DS_Metrics_Reference.ipynb](./DS_Metrics_Reference.ipynb) section 6 (Ranking / IR).

## 6. Modeles 2024-2025 — top embeddings (MTEB leaderboard)

| Modele | Langue | Dims | MTEB |
|---|---|---:|---:|
| `all-MiniLM-L6-v2` | EN | 384 | 56.3 (baseline rapide) |
| `BAAI/bge-large-en-v1.5` | EN | 1024 | 64.2 |
| `intfloat/multilingual-e5-large` | Multi (FR ok) | 1024 | 64.4 |
| `jinaai/jina-embeddings-v3` | Multi | 1024 | 65.5 |
| `BAAI/bge-m3` | Multi (long context 8k) | 1024 | excellent |

## 7. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| BM25 sans tokenization fine | Lowercase + lemmatisation pour mieux matcher |
| Dense seul sans BM25 | Hybrid souvent +5-10% recall |
| Top-k = 1 | Toujours top-3 a top-10 minimum |
| Pas normaliser pour cosine | `q_emb /= norm` |
| Embeddings re-genere a chaque query | Cache / persiste les `doc_embs` |
| Skip re-ranking sur top-100 | Cross-encoder boost typique +10-15% NDCG |


## References

### Documentation
- Sentence-Transformers : https://www.sbert.net/
- rank_bm25 : https://github.com/dorianbrown/rank_bm25
- MTEB leaderboard : https://huggingface.co/spaces/mteb/leaderboard

### Voir aussi
- [BDD_Vectorielles.ipynb](BDD_Vectorielles.ipynb)
- [retrieval_BDD_Vectorielle.ipynb](retrieval_BDD_Vectorielle.ipynb)
- [NLP_LangChain_RAG.ipynb](NLP_LangChain_RAG.ipynb)
- [AI_Prompt_Engineering.ipynb](AI_Prompt_Engineering.ipynb)
- [DS_Metrics_Reference.ipynb](DS_Metrics_Reference.ipynb)
